# 3. Can the cells help tag BIB?

A slow long-lived particle (LLP) and beam-induced background (BIB) can both leave a late jet. Does knowing *where* and *when* the cells were hit help? I'll train four models from the ROOT files and compare them with Emil's timing score.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import time
import copy

import torch
import numpy as np
import pandas as pd
import awkward as ak
import uproot
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import roc_curve, roc_auc_score

In [ ]:
folder = Path.cwd()
project = folder.parent if folder.name == "notebooks" else folder
data_folder = project / "DataForEmre"
assert data_folder.is_dir(), "Open the PPP project and check DataForEmre."
stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
run_dir = project / "analysis/outputs/03_train_and_compare" / stamp
run_dir.mkdir(parents=True, exist_ok=False)
print("Results:", run_dir)

seed = 2026
max_epochs = 20
patience = 5
batch_size = 64
pair_budget = 500_000
learning_rate = 0.001
weight_decay = 0.0001
llp_keep_target = 0.95

torch.set_num_threads(4)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## Where the labels come from

For signal I use simulated jets matched to an LLP decay. For background I use crossings with a bunch in only one beam. These unpaired jets are rich in BIB, though some have other origins.

In [ ]:
llp_files = sorted((data_folder / "collision/LLP/N1N2_rpv_1400_10ns").glob("*.root"))
unpaired_files = sorted((data_folder / "non_collision/unpaired/physics_Background_NTuple_unsplit_v2").glob("*.root"))
vertex_files = sorted((data_folder / "non_collision/unpaired/user.ezaya.UNPAIRED_nPV_nocells.EXT0").glob("*.root"))
sample_files = {"LLP": llp_files, "unpaired": unpaired_files}
assert llp_files and unpaired_files and vertex_files, "A data folder is empty."
display(pd.Series({"LLP files": len(llp_files), "Unpaired cell files": len(unpaired_files),
                   "Vertex files": len(vertex_files)}, name="Files"))

In [ ]:
event_fields = ["runNumber", "eventNumber", "lumiBlock"]
source_rows = []

def note_source(path, sample, entries):
    info = path.stat()
    source_rows.append({
        "sample": sample, "path": str(path.relative_to(data_folder)),
        "entries": entries, "bytes": info.st_size, "mtime_ns": info.st_mtime_ns,
    })

In [ ]:
def add_event_keys(block, seen):
    columns = [block[name] for name in event_fields]
    for run, number, lumi in zip(*columns):
        key = (int(run), int(number))
        if min(*key, int(lumi)) < 0:
            raise ValueError("A cell file has an invalid event ID.")
        if key in seen:
            raise ValueError(f"Repeated event: {key}")
        seen[key] = int(lumi)

def scan_events(files, sample):
    seen = {}
    for path in files:
        with uproot.open(path) as root_file:
            tree = root_file["JetContent"]
            note_source(path, sample, tree.num_entries)
            for block in tree.iterate(event_fields, step_size=100_000, library="np"):
                add_event_keys(block, seen)
    return seen

events = {sample: scan_events(files, sample) for sample, files in sample_files.items()}
display(pd.Series({sample: len(keys) for sample, keys in events.items()}, name="Unique events"))

I get the vertex count from a separate file, matching by run and event number and checking the luminosity block. Keeping zero-vertex unpaired events helps remove collisions.

In [ ]:
vertex_fields = ["runNumber", "eventNumber", "lumiBlockNumber", "nPrimaryVertices"]
vertex_lookup = {}
target_events = events["unpaired"]
target_runs = np.array(sorted({run for run, event in target_events}), dtype=np.int64)
vertex_scan = {"rows": 0, "negative_event_ids_in_target_runs": 0}

In [ ]:
def add_vertices(block):
    vertex_scan["rows"] += len(block["runNumber"])
    relevant = np.isin(block["runNumber"], target_runs)
    runs, event, lumis, counts = (block[name][relevant] for name in vertex_fields)
    # The sidecar stores the 32 event-number bits as a signed integer.
    assert event.dtype == np.dtype("int32"), "Check the sidecar event-number type."
    vertex_scan["negative_event_ids_in_target_runs"] += int((event < 0).sum())
    event = event.astype(np.uint32)
    for run, number, lumi, count in zip(runs, event, lumis, counts):
        key = (int(run), int(number))
        if key not in target_events:
            continue
        if int(lumi) != target_events[key] or count < 0:
            raise ValueError(f"Invalid vertex match: {key}")
        if key in vertex_lookup:
            raise ValueError(f"Repeated vertex match: {key}")
        vertex_lookup[key] = int(count)

In [ ]:
for path in vertex_files:
    with uproot.open(path) as root_file:
        tree = root_file["BIB"]
        note_source(path, "vertex", tree.num_entries)
        for block in tree.iterate(vertex_fields, step_size=100_000, library="np"):
            add_vertices(block)

assert set(vertex_lookup) == set(target_events), "Some cell events lack a vertex match."
print("Matched events:", len(vertex_lookup))
print("Zero-vertex events:", sum(count == 0 for count in vertex_lookup.values()))
print("Sidecar rows scanned:", vertex_scan["rows"])

Both samples need $p_T\geq30$ GeV, $|\eta|<1.7$ and a Tile energy fraction strictly between 0 and 1. I keep early and late jets for training.

The five jet inputs are the ATLAS jet time, its spread and uncertainty, the number of clusters and the Tile energy fraction.

In [ ]:
jet_features = ["jet_time_ATLAS", "jet_time_spread", "jet_time_sigma",
                "jet_n_clusters", "jet_TileEnergyFrac"]
jet_fields = ["jet_pt", "jet_eta", *jet_features]
raw_cell_fields = ["cell_time", "cell_e", "cell_x", "cell_y", "cell_z",
                   "cell_dEta_jet", "cell_dPhi_jet"]
cut_counts = {}

def valid_number(values):
    return np.isfinite(values) & (values != -999999)

def count_cut(sample, stage, keep):
    key = (sample, stage)
    cut_counts[key] = cut_counts.get(key, 0) + int(ak.sum(keep))

In [ ]:
def select_kinematics(batch, sample):
    keep = ak.ones_like(batch["jet_pt"], dtype=bool)
    count_cut(sample, "All jets", keep)
    if sample == "LLP":
        flag = batch["jet_is_llp_matched"]
        assert bool(ak.all((flag == 0) | (flag == 1)))
        keep = keep & (flag == 1)
    count_cut(sample, "LLP matching", keep)
    for name in ["jet_pt", "jet_eta", "jet_time_ATLAS"]:
        keep = keep & valid_number(batch[name])
    count_cut(sample, "Valid pT, eta and time", keep)
    keep = keep & (batch["jet_pt"] / 1000 >= 30)
    count_cut(sample, "pT >= 30 GeV", keep)
    keep = keep & (abs(batch["jet_eta"]) < 1.7)
    count_cut(sample, "abs(eta) < 1.7", keep)
    keep = keep & (batch["jet_TileEnergyFrac"] > 0) & (batch["jet_TileEnergyFrac"] < 1)
    count_cut(sample, "Energy in both calorimeters", keep)
    return keep

In [ ]:
def select_inputs(batch, sample, keep):
    if sample == "unpaired":
        counts = [vertex_lookup[(int(r), int(e))] for r, e in
                  zip(batch["runNumber"], batch["eventNumber"])]
        keep = keep & (ak.Array(counts) == 0)
    count_cut(sample, "Zero vertices for unpaired", keep)
    for name in jet_features:
        keep = keep & valid_number(batch[name])
    keep = keep & (batch["jet_time_spread"] >= 0) & (batch["jet_time_sigma"] > 0)
    n_clusters = batch["jet_n_clusters"]
    keep = keep & (n_clusters >= 1) & (n_clusters == np.floor(n_clusters))
    count_cut(sample, "Valid jet summaries", keep)
    return keep

## From a jet to its cells

The cluster sizes tell me which cells in the event belong to each jet. [Notebook 01 works through one example.](01_cells_and_selections.ipynb)

I keep cells with valid inputs and positive energy. Each gives me a time (ns), radius and $z$ (m), direction relative to the jet $(\Delta\eta,\Delta\phi)$ and a share of the usable cell energy, stored as $\log(1+100 E_i/\sum_j E_j)$.

In [ ]:
def cell_boundaries(row):
    clusters = ak.to_numpy(row["jet_n_clusters"])
    sizes = ak.to_numpy(row["cluster_n_cells"])
    for counts in [clusters, sizes]:
        assert np.isfinite(counts).all() and (counts >= 0).all()
        assert np.equal(counts, np.floor(counts)).all()
    assert len(clusters) == len(row["jet_pt"])
    assert int(clusters.sum()) == len(sizes)
    assert int(sizes.sum()) == len(row["cell_e"])
    cluster_edges = np.r_[0, np.cumsum(clusters, dtype=np.int64)]
    cell_edges = np.r_[0, np.cumsum(sizes, dtype=np.int64)]
    return cell_edges[cluster_edges]

def event_cell_data(row):
    n_jets, n_cells = len(row["jet_pt"]), len(row["cell_e"])
    assert all(len(row[name]) == n_jets for name in jet_fields)
    assert all(len(row[name]) == n_cells for name in raw_cell_fields)
    matrix = np.column_stack([ak.to_numpy(row[name]) for name in raw_cell_fields])
    return matrix, cell_boundaries(row)

In [ ]:
def make_cell_inputs(raw):
    good = valid_number(raw).all(axis=1) & (raw[:, 1] > 0)
    values = raw[good]
    if len(values) == 0:
        return np.empty((0, 6), dtype=np.float32)
    time, energy, x, y, z, deta, dphi = values.T
    relative_energy = np.log1p(100 * energy / energy.sum())
    radius = np.hypot(x, y) / 1000
    inputs = np.column_stack([time, relative_energy, radius, z / 1000, deta, dphi])
    assert np.isfinite(inputs).all()
    return inputs.astype(np.float32)

In [ ]:
def make_jet_record(row, sample, path, row_index, jet_index):
    run, number = int(row["runNumber"]), int(row["eventNumber"])
    record = {"sample": sample, "label": int(sample == "unpaired"),
              "runNumber": run, "eventNumber": number, "lumiBlock": int(row["lumiBlock"]),
              "source_file": str(path.relative_to(data_folder)),
              "row_index": row_index, "jet_index": jet_index,
              "event_group": f"{sample}:{run}:{number}"}
    record.update({name: float(row[name][jet_index]) for name in jet_fields})
    if "jet_BIB_score" in row.fields:
        assert len(row["jet_BIB_score"]) == len(row["jet_pt"])
        score = float(row["jet_BIB_score"][jet_index])
        record["jet_BIB_score"] = score if valid_number(score) else np.nan
    else:
        record["jet_BIB_score"] = np.nan
    record["jet_pt_GeV"] = record["jet_pt"] / 1000
    record["jet_abs_eta"] = abs(record["jet_eta"])
    record["n_vertices"] = vertex_lookup[(run, number)] if sample == "unpaired" else -1
    return record

In [ ]:
jet_records, cell_arrays = [], []
cell_totals = {"stored": 0, "valid": 0, "empty_jets": 0}

def add_selected_event(row, selected, sample, path, row_index):
    raw, edges = event_cell_data(row)
    for jet_index in ak.to_numpy(ak.where(selected)[0]):
        start, stop = edges[jet_index:jet_index + 2]
        cells = make_cell_inputs(raw[start:stop])
        cell_totals["stored"] += int(stop - start)
        cell_totals["valid"] += len(cells)
        if not len(cells):
            cell_totals["empty_jets"] += 1
            continue
        record = make_jet_record(row, sample, path, row_index, int(jet_index))
        record["n_cells"] = len(cells)
        jet_records.append(record)
        cell_arrays.append(cells)
        key = (sample, "At least one valid cell")
        cut_counts[key] = cut_counts.get(key, 0) + 1

In [ ]:
def read_cell_file(path, sample):
    fields = [*event_fields, *jet_fields, "cluster_n_cells", *raw_cell_fields]
    if sample == "LLP":
        fields.append("jet_is_llp_matched")
    offset = 0
    with uproot.open(path) as root_file:
        tree = root_file["JetContent"]
        if "jet_BIB_score" in tree:
            fields.append("jet_BIB_score")
        for batch in tree.iterate(fields, step_size=1000, library="ak"):
            keep = select_kinematics(batch, sample)
            keep = select_inputs(batch, sample, keep)
            selected_rows = ak.to_numpy(ak.where(ak.any(keep, axis=1))[0])
            for i in selected_rows:
                add_selected_event(batch[i], keep[i], sample, path, offset + int(i))
            offset += len(batch)
    return offset

In [ ]:
for sample, files in sample_files.items():
    read_events = 0
    for i, path in enumerate(files, start=1):
        read_events += read_cell_file(path, sample)
        if i == len(files) or i % 50 == 0:
            print(sample, "files read:", i, "of", len(files))
    assert read_events == len(events[sample])

jets = pd.DataFrame(jet_records).reset_index(drop=True)
del jet_records
assert len(jets) == len(cell_arrays)
assert not jets.duplicated(["event_group", "jet_index"]).any()
print("Selected jets:", len(jets), "  Valid cells:", cell_totals["valid"])

Read down the table to see what each cut removes. The zero-vertex requirement takes the unpaired sample from 17,223 jets to 3,463.

In [ ]:
stages = ["All jets", "LLP matching", "Valid pT, eta and time", "pT >= 30 GeV",
          "abs(eta) < 1.7", "Energy in both calorimeters", "Zero vertices for unpaired",
          "Valid jet summaries", "At least one valid cell"]
cutflow = pd.DataFrame({sample: [cut_counts.get((sample, stage), 0) for stage in stages]
                       for sample in sample_files}, index=stages)
for sample in sample_files:
    cutflow[f"{sample} removed"] = -cutflow[sample].diff().fillna(0).astype(int)
display(cutflow)
print("Jets with no valid cells:", cell_totals["empty_jets"])

In [ ]:
source_table = pd.DataFrame(source_rows)
source_table.to_csv(run_dir / "sources.csv", index=False)
cutflow.to_csv(run_dir / "cutflow.csv")
data_audit = {"source_files": len(source_table), "selected_jets": len(jets),
              "cells": cell_totals, "vertex_scan": vertex_scan,
              "matched_vertex_events": len(vertex_lookup),
              "zero_vertex_events": sum(n == 0 for n in vertex_lookup.values())}
(run_dir / "data_audit.json").write_text(json.dumps(data_audit, indent=2))
display(jets.groupby("sample").agg(jets=("label", "size"), cells=("n_cells", "sum")))

## Split by event

I split events 60/20/20 for training, validation and testing. Jets from one event stay together. I scale inputs with the training mean and standard deviation, $x'=(x-\mu)/\sigma$. Equal total weight per class keeps the larger LLP sample from dominating training.

In [ ]:
rng = np.random.default_rng(seed)
event_split = {}
for sample in ["LLP", "unpaired"]:
    groups = np.sort(jets.loc[jets["sample"].eq(sample), "event_group"].unique())
    groups = rng.permutation(groups)
    first, second = int(0.6 * len(groups)), int(0.8 * len(groups))
    pieces = [groups[:first], groups[first:second], groups[second:]]
    for split, part in zip(["train", "validation", "test"], pieces):
        event_split.update({group: split for group in part})
jets["split"] = jets["event_group"].map(event_split)

assert jets["split"].notna().all()
assert jets.groupby("event_group")["split"].nunique().eq(1).all()
split_rows = {
    name: np.flatnonzero(jets["split"].eq(name))
    for name in ["train", "validation", "test"]
}
split_counts = jets.groupby(["split", "sample"]).agg(
    jets=("label", "size"), events=("event_group", "nunique"),
)
display(split_counts)
jets.to_csv(run_dir / "selected_jets_with_split.csv.gz", index=False)

In [ ]:
labels_array = jets["label"].to_numpy(dtype=np.int64)
jet_array = jets[jet_features].to_numpy(dtype=np.float32)
cell_lengths = np.array([len(array) for array in cell_arrays])
# ParT measures distances and time differences before the inputs are scaled.
geometry_arrays = [
    torch.from_numpy(array[:, [4, 5, 0, 2, 3]].copy()) for array in cell_arrays
]
assert (cell_lengths > 0).all()
assert np.isfinite(jet_array).all()
print("Jets:", len(jets), "Cells:", cell_lengths.sum())

In [ ]:
train_rows = split_rows["train"]
training_cells = np.concatenate([cell_arrays[i] for i in train_rows])
jet_mean = jet_array[train_rows].mean(axis=0, dtype=np.float64).astype(np.float32)
jet_scale = jet_array[train_rows].std(axis=0, dtype=np.float64).astype(np.float32)
cell_mean = training_cells.mean(axis=0, dtype=np.float64).astype(np.float32)
cell_scale = training_cells.std(axis=0, dtype=np.float64).astype(np.float32)
jet_scale[jet_scale == 0] = 1
cell_scale[cell_scale == 0] = 1
del training_cells

In [ ]:
jet_tensor = torch.from_numpy((jet_array - jet_mean) / jet_scale)
cell_tensors = [torch.from_numpy((array - cell_mean) / cell_scale) for array in cell_arrays]
target_tensor = torch.from_numpy(labels_array.astype(np.float32))
scaling = {"jet_mean": jet_mean.tolist(), "jet_scale": jet_scale.tolist(),
           "cell_mean": cell_mean.tolist(), "cell_scale": cell_scale.tolist()}
(run_dir / "scaling.json").write_text(json.dumps(scaling, indent=2))

class_counts = np.bincount(labels_array[train_rows], minlength=2)
assert (class_counts > 0).all()
class_weights = len(train_rows) / (2 * class_counts)
weight_tensor = torch.from_numpy(class_weights[labels_array].astype(np.float32))
display(pd.DataFrame({"Class": ["LLP", "unpaired"],
                      "Training jets": class_counts, "Weight": class_weights}))

I keep all usable cells. I group jets of similar size and mask the padding. When cell pairs take too much memory, I split the batch into pieces and update the model once.

In [ ]:
def make_batches(rows, shuffle_seed=None):
    ordered = np.asarray(rows)[np.argsort(-cell_lengths[rows], kind="stable")]
    batches = [ordered[i:i + batch_size] for i in range(0, len(ordered), batch_size)]
    if shuffle_seed is not None:
        np.random.default_rng(shuffle_seed).shuffle(batches)
    return batches

check_rows = np.concatenate(make_batches(train_rows))
np.testing.assert_array_equal(np.sort(check_rows), np.sort(train_rows))

In [ ]:
def memory_batches(rows):
    ordered = np.asarray(rows)[np.argsort(-cell_lengths[rows], kind="stable")]
    start = 0
    while start < len(ordered):
        largest = int(cell_lengths[ordered[start]])
        count = max(1, min(batch_size, pair_budget // largest**2))
        yield ordered[start:start + count]
        start += count

chunks = np.concatenate(list(memory_batches(train_rows)))
np.testing.assert_array_equal(np.sort(chunks), np.sort(train_rows))

In [ ]:
from torch.nn.utils.rnn import pad_sequence

def pack_batch(rows):
    cells = pad_sequence([cell_tensors[i] for i in rows], batch_first=True)
    geometry = pad_sequence([geometry_arrays[i] for i in rows], batch_first=True)
    counts = torch.as_tensor(cell_lengths[rows])
    valid = torch.arange(cells.shape[1])[None, :] < counts[:, None]
    inputs = (jet_tensor[rows], cells, valid, geometry)
    inputs = tuple(item.to(device) for item in inputs)
    return inputs, target_tensor[rows].to(device), weight_tensor[rows].to(device)

## Start with the jet, then add the cells

The **jet MLP** sees five numbers per jet. Each cell model gets the same five plus a summary of the cells.

**Deep Sets** runs one network $\phi$ on every cell and averages the outputs, $h=\frac{1}{n}\sum_i\phi(x_i)$. Shuffling the cells leaves $h$ unchanged.

In [ ]:
from torch import nn
from torch.nn import functional as F

width = 32
heads = 4
jet_dim = len(jet_features)
cell_dim = 6

def decision_head(input_size):
    return nn.Sequential(
        nn.Linear(input_size, width),
        nn.GELU(),
        nn.Dropout(0.1),
        nn.Linear(width, 1),
    )

def masked_average(values, valid):
    values = values.masked_fill(~valid.unsqueeze(-1), 0)
    count = valid.sum(dim=1, keepdim=True).clamp_min(1)
    return values.sum(dim=1) / count

In [ ]:
class JetCellModel(nn.Module):
    def __init__(self, encoder=None):
        super().__init__()
        self.encoder = encoder
        input_size = jet_dim + (width if encoder is not None else 0)
        self.head = decision_head(input_size)

    def forward(self, jets, cells, valid, geometry):
        if self.encoder is None:
            return self.head(jets).squeeze(-1)
        if not bool(valid.any(dim=1).all()):
            raise ValueError("A cell model needs at least one valid cell per jet.")
        cells = cells.masked_fill(~valid.unsqueeze(-1), 0)
        geometry = geometry.masked_fill(~valid.unsqueeze(-1), 0)
        summary = self.encoder(cells, valid, geometry)
        combined = torch.cat([jets, summary], dim=1)
        return self.head(combined).squeeze(-1)

In [ ]:
class DeepSetsEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.read_cell = nn.Sequential(
            nn.Linear(cell_dim, width), nn.GELU(),
            nn.Linear(width, width), nn.GELU(),
        )

    def forward(self, cells, valid, geometry):
        encoded = self.read_cell(cells)
        return masked_average(encoded, valid)

### Set Transformer

Sixteen learned vectors gather information from the cells using attention. Each cell then reads those summaries before I take the average. A cell can now use information from elsewhere in the jet. [Set Transformer paper](https://proceedings.mlr.press/v97/lee19d.html)

In [ ]:
class CrossAttentionBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.attention = nn.MultiheadAttention(width, heads, dropout=0.1, batch_first=True)
        self.norm1 = nn.LayerNorm(width)
        self.norm2 = nn.LayerNorm(width)
        self.update = nn.Sequential(
            nn.Linear(width, 2 * width), nn.GELU(),
            nn.Dropout(0.1), nn.Linear(2 * width, width),
        )

    def forward(self, queries, information, valid_keys=None):
        padding = None if valid_keys is None else ~valid_keys
        message, _ = self.attention(
            queries, information, information,
            key_padding_mask=padding, need_weights=False,
        )
        state = self.norm1(queries + message)
        return self.norm2(state + self.update(state))

In [ ]:
class SetTransformerEncoder(nn.Module):
    def __init__(self, inducing_count=16):
        super().__init__()
        self.project = nn.Linear(cell_dim, width)
        self.inducing = nn.Parameter(torch.randn(1, inducing_count, width) * 0.1)
        self.gather = CrossAttentionBlock()
        self.distribute = CrossAttentionBlock()

    def forward(self, cells, valid, geometry):
        encoded = self.project(cells)
        queries = self.inducing.expand(cells.shape[0], -1, -1)
        summaries = self.gather(queries, encoded, valid)
        updated = self.distribute(encoded, summaries)
        return masked_average(updated, valid)

### Small ParT-inspired model

I compare every cell pair. Their angular distance and differences in time, radius and $z$ give an extra term $b_{ij}$ in attention. This lets a cell use the timing of another cell further along the beam. I use one attention block. [Particle Transformer paper](https://proceedings.mlr.press/v162/qu22b.html)

In [ ]:
def pair_features(geometry):
    difference = geometry.unsqueeze(2) - geometry.unsqueeze(1)
    deta = difference[..., 0]
    dphi = (difference[..., 1] + torch.pi) % (2 * torch.pi) - torch.pi
    angular_distance = torch.sqrt(deta.square() + dphi.square())
    time_difference = difference[..., 2].abs() / 10
    radial_difference = difference[..., 3].abs()
    z_difference = difference[..., 4].abs()
    features = torch.stack([
        angular_distance, time_difference, radial_difference, z_difference,
    ], dim=-1)
    return torch.log1p(features)

In [ ]:
class PairAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.qkv = nn.Linear(width, 3 * width)
        self.pair_bias = nn.Linear(4, heads, bias=False)
        self.output = nn.Linear(width, width)

    def forward(self, encoded, valid, geometry):
        batch, count, _ = encoded.shape
        qkv = self.qkv(encoded).reshape(batch, count, 3, heads, width // heads)
        query, key, value = qkv.permute(2, 0, 3, 1, 4).unbind(0)
        bias = self.pair_bias(pair_features(geometry)).permute(0, 3, 1, 2)
        bias = bias.masked_fill(~valid[:, None, None, :], -torch.inf)
        attended = F.scaled_dot_product_attention(
            query, key, value, attn_mask=bias,
            dropout_p=0.1 if self.training else 0.0,
        )
        attended = attended.transpose(1, 2).reshape(batch, count, width)
        return self.output(attended)

In [ ]:
class PairTransformerEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.project = nn.Linear(cell_dim, width)
        self.attention = PairAttention()
        self.norm1 = nn.LayerNorm(width)
        self.norm2 = nn.LayerNorm(width)
        self.update = nn.Sequential(
            nn.Linear(width, 2 * width), nn.GELU(),
            nn.Dropout(0.1), nn.Linear(2 * width, width),
        )

    def forward(self, cells, valid, geometry):
        encoded = self.project(cells)
        state = self.norm1(encoded + self.attention(encoded, valid, geometry))
        state = self.norm2(state + self.update(state))
        return masked_average(state, valid)

In [ ]:
model_factories = {
    "Jet MLP": lambda: JetCellModel(),
    "Deep Sets": lambda: JetCellModel(DeepSetsEncoder()),
    "Set Transformer": lambda: JetCellModel(SetTransformerEncoder()),
    "ParT-inspired": lambda: JetCellModel(PairTransformerEncoder()),
}

## Train the four models

The output $z$ gives a BIB score $s=1/(1+e^{-z})$. Low scores for unpaired jets and high scores for LLP jets get penalized. I keep the epoch with the lowest validation loss. I stop after five epochs without improvement or 20 in total.

In [ ]:
def train_epoch(model, optimizer, epoch):
    model.train()
    for rows in make_batches(train_rows, seed + epoch):
        optimizer.zero_grad(set_to_none=True)
        for part in memory_batches(rows):
            inputs, targets, weights = pack_batch(part)
            logits = model(*inputs)
            errors = F.binary_cross_entropy_with_logits(
                logits, targets, reduction="none",
            )
            loss = (weights * errors).sum() / len(rows)
            loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()

In [ ]:
@torch.inference_mode()
def predict_logits(model, rows):
    model.eval()
    result = np.full(len(jets), np.nan, dtype=np.float64)
    for batch in memory_batches(rows):
        inputs, _, _ = pack_batch(batch)
        result[batch] = model(*inputs).cpu().numpy().astype(np.float64)
    assert np.isfinite(result[rows]).all()
    return result[rows]

In [ ]:
def weighted_loss(raw_logits, rows):
    targets = labels_array[rows]
    weights = class_weights[targets]
    errors = np.logaddexp(0, raw_logits) - targets * raw_logits
    return float(np.sum(weights * errors) / weights.sum())

def assess(model, rows):
    raw_logits = predict_logits(model, rows)
    return weighted_loss(raw_logits, rows)

In [ ]:
def keep_best(state, model, validation_loss, epoch):
    if validation_loss < state["loss"]:
        state.update(loss=validation_loss, epoch=epoch, wait=0,
                     weights=copy.deepcopy(model.state_dict()))
    else:
        state["wait"] += 1
    return state["wait"] >= patience

def fit(model, optimizer):
    state = {"loss": float("inf"), "wait": 0}
    history = []
    for epoch in range(1, max_epochs + 1):
        train_epoch(model, optimizer, epoch)
        train_loss = assess(model, split_rows["train"])
        validation_loss = assess(model, split_rows["validation"])
        history.append({"epoch": epoch, "train_loss": train_loss,
                        "validation_loss": validation_loss})
        print(f"Epoch {epoch:2d}: train {train_loss:.4f}, validation {validation_loss:.4f}", flush=True)
        if keep_best(state, model, validation_loss, epoch):
            break
    model.load_state_dict(state["weights"])
    return history, state["epoch"]

In [ ]:
models, histories, model_details = {}, {}, {}
for name, factory in model_factories.items():
    print("Training:", name, flush=True)
    torch.manual_seed(seed)
    model = factory().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    started = time.perf_counter()
    history, selected_epoch = fit(model, optimizer)
    model_details[name] = {"parameters": sum(p.numel() for p in model.parameters()),
                          "train_seconds": time.perf_counter() - started,
                          "selected_epoch": selected_epoch, "epochs_run": len(history)}
    models[name], histories[name] = model.eval(), history

## How many delayed jets get tagged?

Here I keep jets with ATLAS time **above 3 ns**. A score at or above the cut tags a jet as BIB. I count unpaired jets tagged and LLP jets kept.

In [ ]:
score_tables = {}
for split in ["validation", "test"]:
    rows = split_rows[split]
    frame = jets.iloc[rows].reset_index(drop=True).copy()
    for name, model in models.items():
        logits = predict_logits(model, rows)
        frame[name] = torch.from_numpy(logits).sigmoid().numpy()
    frame["Emil"] = frame["jet_BIB_score"]
    frame["delayed"] = frame["jet_time_ATLAS"] > 3
    score_tables[split] = frame

score_tables["test"].to_csv(run_dir / "test_jet_scores.csv.gz", index=False)
validation = score_tables["validation"]
test = score_tables["test"]
delayed_validation = validation.loc[validation["delayed"]]
delayed_test = test.loc[test["delayed"]]

For each model I sort delayed validation LLP jets from lowest to highest score. I put the cut just above the score where I reach 95%, keeping ties together. Then I count the test jets at that cut.

In [ ]:
def retention_cut(values, weights, target=llp_keep_target):
    positive = weights > 0
    if not positive.any():
        raise ValueError("A retention cut needs LLP jets with positive weight.")
    order = np.argsort(values[positive], kind="stable")
    sorted_scores = values[positive][order]
    cumulative = np.cumsum(weights[positive][order], dtype=np.float64)
    edge = np.searchsorted(cumulative, target * cumulative[-1], side="left")
    return float(np.nextafter(sorted_scores[edge], np.inf))

def count_decisions(frame, method, cut):
    tagged = frame[method].to_numpy() >= cut
    unpaired = frame["label"].eq(1).to_numpy()
    return {"Method": method, "Cut": cut,
            "Unpaired tagged": int(tagged[unpaired].sum()),
            "Unpaired total": int(unpaired.sum()),
            "LLP kept": int((~tagged[~unpaired]).sum()),
            "LLP total": int((~unpaired).sum())}

In [ ]:
llp_validation = delayed_validation.loc[delayed_validation["label"].eq(0)]
plain_cuts = {name: retention_cut(llp_validation[name].to_numpy(), np.ones(len(llp_validation)))
              for name in models}
rows = [count_decisions(delayed_test, name, cut) for name, cut in plain_cuts.items()]
model_table = pd.DataFrame(rows).set_index("Method").join(pd.DataFrame(model_details).T)
display(model_table[["Unpaired tagged", "Unpaired total", "LLP kept", "LLP total",
                     "selected_epoch", "train_seconds"]].round(2))

The dashed line marks the epoch I kept. For ParT that's epoch 14. From epoch 14 to 19 the training loss falls from 0.036 to 0.032, while the validation loss stays around 0.040. The extra training helps on jets the model has already seen.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(9, 5))
for ax, (name, history) in zip(axes.flat, histories.items()):
    curve = pd.DataFrame(history)
    ax.plot(curve["epoch"], curve["train_loss"], label="Train")
    ax.plot(curve["epoch"], curve["validation_loss"], label="Validation")
    ax.axvline(model_details[name]["selected_epoch"], color="0.5", linestyle="--")
    ax.set(title=name, xlabel="Epoch", ylabel="Weighted loss")
axes[0, 0].legend()
fig.tight_layout()
fig.savefig(run_dir / "learning_curves.png", dpi=150)
plt.show()

## ParT and Emil on exactly the same jets

Emil's score compares a beam-like timing pattern with a flat one. Three delayed unpaired test jets lack his score. That leaves **44 unpaired and 803 LLP** jets for both methods.

I balance the samples in bins of $p_T$ and $|\eta|$, separately for validation and test. Both methods use the same weights. In a bin with 10 LLP and 2 unpaired jets, each LLP gets weight $2/10$ and each unpaired jet gets 1. Bins with only one class get zero weight. Raw counts include those jets.

In [ ]:
pt_edges = [30, 60, 100, 200, 400, 800, 1600, np.inf]
eta_edges = [0, 0.5, 1.0, 1.7]
methods = ["ParT-inspired", "Emil"]

def with_overlap_weights(frame):
    common = frame.loc[np.isfinite(frame[methods]).all(axis=1)].reset_index(drop=True).copy()
    common["pt_bin"] = pd.cut(common["jet_pt_GeV"], pt_edges, right=False)
    common["eta_bin"] = pd.cut(common["jet_abs_eta"], eta_edges, right=False)
    common["weight"] = 0.0
    for _, group in common.groupby(["pt_bin", "eta_bin"], observed=True):
        counts = group["label"].value_counts().reindex([0, 1], fill_value=0)
        if counts.min() > 0:
            common.loc[group.index, "weight"] = counts.min() / group["label"].map(counts)
    return common

common_val = with_overlap_weights(delayed_validation)
common_test = with_overlap_weights(delayed_test)
assert common_val["weight"].sum() > 0 and common_test["weight"].sum() > 0
display(common_test.groupby("sample").agg(
    jets=("label", "size"),
    jets_with_positive_weight=("weight", lambda w: (w > 0).sum()),
    total_weight=("weight", "sum"),
))

I choose each cut to keep at least **95% of the validation LLP weight**, then freeze it for testing. Losing a few test jets with large weights can make the weighted retention drop well below the raw fraction kept.

In [ ]:
def cut_for_llp(frame, method):
    llp = frame.loc[frame["label"].eq(0)]
    return retention_cut(llp[method].to_numpy(), llp["weight"].to_numpy())

def weighted_result(frame, method, cut):
    row = count_decisions(frame, method, cut)
    tagged = frame[method].to_numpy() >= cut
    unpaired = frame["label"].eq(1).to_numpy()
    weights = frame["weight"].to_numpy()
    row["Weighted unpaired tagged (%)"] = 100 * np.average(tagged[unpaired], weights=weights[unpaired])
    row["Weighted LLP kept (%)"] = 100 * np.average(~tagged[~unpaired], weights=weights[~unpaired])
    return row

validation_cuts = {name: cut_for_llp(common_val, name) for name in methods}
calibration = [weighted_result(common_val, name, cut) for name, cut in validation_cuts.items()]
display(pd.DataFrame(calibration).set_index("Method")[["Cut", "Weighted LLP kept (%)"]].round(3))
rows = [weighted_result(common_test, name, cut) for name, cut in validation_cuts.items()]
validation_result = pd.DataFrame(rows).set_index("Method")
display(validation_result.round(3))
validation_result.to_csv(run_dir / "weighted_delayed_test_comparison.csv")

### Reading the weighted ROC

Going right means losing more LLP weight. Going up means tagging more unpaired weight. The dashed line marks 5% LLP loss. The dots use cuts picked on this **test curve**. The table above uses cuts fixed on **validation**.

In [ ]:
roc_cuts = {name: cut_for_llp(common_test, name) for name in methods}
roc_table = pd.DataFrame([weighted_result(common_test, name, cut)
                          for name, cut in roc_cuts.items()]).set_index("Method")
display(roc_table.round(3))
roc_table.to_csv(run_dir / "weighted_delayed_test_roc_scan.csv")

fig, ax = plt.subplots(figsize=(7, 4))
for name in methods:
    x, y, _ = roc_curve(common_test["label"], common_test[name], sample_weight=common_test["weight"])
    auc = roc_auc_score(common_test["label"], common_test[name], sample_weight=common_test["weight"])
    line, = ax.plot(100*x, 100*y, label=f"{name} (AUC {auc:.3f})")
    result = roc_table.loc[name]
    ax.scatter(100 - result["Weighted LLP kept (%)"], result["Weighted unpaired tagged (%)"],
               color=line.get_color(), zorder=3)
ax.axvline(100 * (1 - llp_keep_target), color="0.4", linestyle="--")
ax.set(xlabel="Weighted LLP lost (%)", ylabel="Weighted unpaired tagged (%)", xlim=(0, 15), ylim=(0, 102))
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(run_dir / "weighted_delayed_roc.png", dpi=150)
plt.show()

### What I got on run 484258

Both methods saw the same simulated LLP jets and the same unpaired jets from run 484258.

| Cut chosen on | Method | Unpaired tagged | LLP kept | Weighted LLP kept |
| --- | --- | ---: | ---: | ---: |
| Validation | ParT-inspired | 41/44 | 792/803 | 84.68% |
| Validation | Emil | 5/44 | 784/803 | 97.12% |
| Test ROC | ParT-inspired | 40/44 | 798/803 | 95.29% |
| Test ROC | Emil | 13/44 | 774/803 | 95.00% |

ParT tags more unpaired jets at similar test retention. Its validation cut missed the 95% target on test, though. I'd check that cut on cells from a second unpaired run.

In [ ]:
for name, model in models.items():
    checkpoint = {"model": name, "weights": model.state_dict(), "scaling": scaling,
                  "jet_inputs": jet_features, "delayed_cut": plain_cuts[name]}
    if name in validation_cuts:
        checkpoint["weighted_validation_cut"] = validation_cuts[name]
    torch.save(checkpoint, run_dir / f"{name.replace(' ', '_')}.pt")
print("Saved:", run_dir)